# LINDAS Tutorial

<img src="https://lindas.admin.ch/static-assets/img/lindaslogo.png" style="width:25%; float:right; padding:20px">

Dieses Tutorial ist dazu gedacht, eine Einführung in das Linked Data Ökosystem LINDAS der schweizerischen Eigenossenschaft zu geben.

## Lernziele

- Sie verstehen in Grundzügen, was Linked Data ist
- Sie kennen das Datenformat RDF
- Sie kennen URI als Identifier für Objekte
- Sie wissen, was SPARQL Queries sind
- Sie verstehen, was LINDAS ist
- Sie können Daten in LINDAS finden
- Sie haben einen Überblick über die verfügbaren Daten in LINDAS
- Sie können SPARQL Queries auf LINDAS ausführen

## Interaktives Notebook

<img src="https://jupyter.org/assets/homepage/main-logo.svg" style="width:15%; float:left; padding:20px">

Dieses Tutorial ist ein sogenanntes **interaktives JupyterLite Notebook**. In diesem Notebook können Sie den Inhalt der einzelnen Zellen interaktiv ändern und diese Zellen direkt ausführen, um das Ergebnis Ihrer Änderungen sofort zu sehen. Die Zellen enthalten entweder [Markdown](https://en.wikipedia.org/wiki/Markdown)-Inhalt (wie diese Zelle) oder ausführbaren Python-Quellcode. Dies ist für ein Tutorial sehr hilfreich, weil Inhalte beliebig mit ausführbarem Quellcode kombiniert werden können. Es können also Abfragen gezeigt werden, diese erklärt werden und darauf weiter aufgebaut werden.

- **Um direkt loslegen zu können klicken Sie oben im Menu auf Run -> Run All Cells.**  
- **Einzelne ausgewählte Zellen können sie danach mit dem "Play-Button" erneut ausführen und so Abfragen individuell anpassen.**

Das Notebook startet mit einem [Setup](#Setup) der Programierumgebung. Das eigentliche Tutorial startet [hier](#Linked-Data-Einführung).

*Zusätzliche Informationen zu JupyterLite:*  
JupyterLite is ein spezielles Jupyter Notebook mit dem Vorteil, vollständig browserbasiert zu sein, ohne eine aufwändige Backend-Infrastruktur zu benötigen. Der Nachteil ist, dass die erstmalige Ausführung der Zellen einige Zeit in Anspruch nehmen kann, weil eine erhebliche Menge von Daten geladen werden muss. Nachfolgende Ausführungen werden aufgrund der gespeicherten Daten in Ihrem Browser-Cache viel schneller sein.

Wenn Sie mehr über die Handhabung von Jupyter Notebooks wissen wollen, finden Sie hier zwei nützliche Ressourcen:

- [Die JupyterLab Interface](https://jupyterlab.readthedocs.io/en/stable/user/interface.html)
- [Das Jupyter Notebook](https://jupyterlab.readthedocs.io/en/stable/user/notebook.html)


## Setup

Eine SPARQL Abfrage ist nichts anderes als ein POST-Request an den entsprechenden Triple Store Datenbank Server. Um diese Requests und die erhaltenen Antworten einfacher handhaben zu können, enthält dieses Notebook vorbereiteten Python Code, der nachfolgend importiert wird. Zusätzlich werden verschiedene Module installiert resp. importiert - u.a. wird das `pandas` Modul importiert, welches die Möglichkeit bietet, die tabellarischen Daten der SPARQL Abfragen als [Pandas Dataframes](https://pandas.pydata.org/docs/index.html) weiter zu verarbeiten. 

In [ ]:
# Installation von folium, um Daten auf Landkarten darzustellen
%pip install -q folium

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
import branca
from ext.sparql import query, display_result

# Linked Data Einführung

Linked Data beschreibt ein **Framework für den Umgang mit Daten**, die sowohl für Menschen nützlich sein sollen, als auch maschinenlesbar sind inkl. einer von Computern verarbeitbaren Semantik. Also sowohl Menschen als auch Computer sollen die Daten "verstehen" und interpretieren können. 

## RDF

Das für Linked Data verwendete Datenformat ist RDF (Resource Description Framework). Das bedeutet, dass die Daten nicht als Tabellen (wie beispielsweise in relationalen Datenbanken) sondern als **Triples** abgespeichert werden. Triples folgen der grammatikalischen Struktur **Subjekt -> Prädikat -> Objekt** und können auch als grammatikalischer Satz verstanden werden. 

Die Information "**Der Apfel ist grün**" wird also mit dem Tripel **Apfel -> ist -> grün** ausgedrückt. Alle Teile eines Triples sind dabei durch weitere Eigenschaften definiert und beschrieben die wiederum in Form von Triples beschrieben sind. Diese vielseitigen Verknüpfungen führen zu einer Netzwerkstruktur, zu einem sogenannten Graphen.

Nachfolgendes Bild aus dem [W3C Primer für RDF](https://www.w3.org/TR/rdf11-primer/) veranschaulicht diese Zusammenhänge:

![](https://www.w3.org/TR/rdf11-primer/example-graph.jpg)

## URI

Eine weitere wichtige Eigenschaft von Linked Data ist, dass alle Teile eines Triples, also Subjekt, Prädikat und Objekt weltweit eindeutig identifizierbar sind. Dafür werden URI (Universal Resource Identifier) eingesetzt. Die URI https://ld.admin.ch/municipality/351 beispielsweise ist der weltweit eindeutige Identifier für die Stadt Bern (natürlich existieren an anderen Orten noch weitere Identifier für die Stadt Bern). Typischerweise lassen sich URI **dereferenzieren**, das heisst, ein Request auf die entsprechende URI (bspw. in dem man sie in die Adresszeile eines Browsers eintippt) führt zu einer Website, die Infos zur entsprechenden URI enthält. Im Falle der URI der Stadt Bern wird man auf eine Webpage weitergeleitet, die diverse Informationen zur Stadt Bern enthält. Diese Informationen sind exakt diejenigen Triples in LINDAS, deren Subjekt die Stadt Bern ist.

## SPARQL

<img src="https://cygri.github.io/rdf-logos/svg/sparql.svg" style="width:20%; float:right; padding:20px">

SPARQL ist eine Query-Sprache für Linked Data Triple Stores. Für eine allgemeine Einführung in SPARQL siehe z.B.: https://en.wikibooks.org/wiki/SPARQL. 

Abfragen (engl. Queries) können entweder direkt über das [SPARQL-Interface von LINDAS](https://ld.admin.ch/sparql) eingegeben und ausgeführt werden, oder als HTTP-POST Request an den SPARQL-Endpoint (https://ld.admin.ch/query) gesendet werden.

Die letztere Methode erlaubt es, eigene Anwendungen zu bauen, die automatisch aktuelle Daten von LINDAS abfragen können. Für dieses Tutorial verwenden wir diese Methode. Die Abfragen sind jedoch in beiden Fällen identisch. Für die Abfrage über das Interface kopieren Sie einfach den Teil zwischen den `"""` und fügen sie im SPARQL-Interface ein.

### Pattern Matching

SPARQL Queries sind Aufträge an den Computer, bestimme Muster (Pattern) in den Daten zu finden (matching). Es können also mit Hilfe von SPARQL Muster vorgegeben werden, und die Datenbank gibt alle Triples zurück, die dieses Muster erfüllen. Einzelne Positionen der Triples können dabei bei einer Abfrage bewusst undefiniert gelassen und mit einer Variable bezeichnet werden. Variablen beginnen mit `?` und werden bei der Abfrage durch alle möglichen Elemente gefüllt, die dieses Pattern erfüllen. Eine ausführlichere Anleitung zum Pattern Matching ist [hier](https://programminghistorian.org/en/lessons/retired/graph-databases-and-SPARQL#rdf-in-brief) zu finden. 

## Weitere Informationen zu Linked Data

Wer vertieft in das Thema Linked Data einsteigen möchte, dem sei beispielsweise [diese Youtube Playlist](https://www.youtube.com/watch?v=ON0wf0SEPx8&list=PLoOmvuyo5UAfY6jb46jCpMoqb-dbVewxg) empfohlen.

# Datasets

Daten in LINDAS sind in Datasets gruppiert. Diese fassen gleichartige Daten zusammen. Datasets in LINDAS haben den [rdf:type](http://www.w3.org/1999/02/22-rdf-syntax-ns#type) [schema:Dataset](https://schema.org/Dataset).

## Alle Datasets

Die erste SPARQL Query dient dazu, herauszufinden, welche Datasets in LINDAS aktuell existieren:

In [ ]:
df = await query("""

SELECT * WHERE {
    
    # suche alle Subjekte, der Prädikate und Objekte die entsprechenden Werte haben
    ?dataset <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://schema.org/Dataset>.

} LIMIT 10 

""")

display_result(df)

Diese URI der Datasets können angeklickt werden und damit öffnet sich eine Website mit weiteren Informationen zum entsprechenden Dataset.

Um, die SPARQL Queries übersichtlicher zu gestalten, können häufig gebrauchte URI mit Prefixes abgekürzt werden. Ausserdem existiert die generische Abkürzung 'a' für http://www.w3.org/1999/02/22-rdf-syntax-ns#type. Damit wird die oben gezeigte Query zu:

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>

SELECT * WHERE {
    
    # suche alle Subjekte, der Prädikate und Objekte die entsprechenden Werte haben
    ?dataset a schema:Dataset.

} LIMIT 10 

""")

display_result(df)

Um etwas mehr, als nur die URI der Datasets als Resultat zu erhalten, kann gleichzeitig auch nach dem Namen des Datasets gefragt werden. Da die Namen in verschiedenen Sprachen angegeben sind, muss nach der gewünschten Sprache gefiltert werden, ansonsten würde man jedes Dataset in jeder angegebenen Sprache zurückerhalten:

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>

SELECT * WHERE {
    ?dataset a schema:Dataset;
        schema:name ?name.
        
    FILTER(lang(?name) = 'de')
}
ORDER BY (?dataset)

""")

display_result(df)

Es zeigt sich, dass diverse Datasets in verschiedenen Versionen existieren. Dasjenige mit der grössten Versionsnummer das aktuellste. Das bedeutet nicht unbedingt, dass es sich um aktuellere Daten handelt, eher noch, dass die Metadaten in besserer und vollständigeren Qualität vorliegen.

Ein Klick bspw. auf https://environment.ld.admin.ch/foen/ubd000504/3 öffnet die entsprechende Website mit weiteren Informationen zum Dataset, bspw. ist dort angegeben, dass der [purl:creator](http://purl.org/dc/terms/creator) des Datensatzes folgendende URI ist: https://register.ld.admin.ch/opendataswiss/org/bundesamt-fur-umwelt-bafu. Es handelt sich also um einen Datensatz des Bundesamts für Umwelt.

## Alle Datenlieferanten

Die Datenlieferanten der Datasets sind über [purl:creator](http://purl.org/dc/terms/creator) angehängt. Folgende SPARQL Query liefert alle Datenlieferanten in LINDAS:

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>

SELECT DISTINCT ?owner ?name WHERE {
    ?dataset a schema:Dataset;
        purl:creator ?owner.
    
    OPTIONAL {
    ?owner schema:name ?name.
    
    FILTER(lang(?name) = "de")
    }
}

""")

display_result(df)

## Datasets eines spezifischen Datenlieferanten

Das Prädikat [purl:creator](http://purl.org/dc/terms/creator) kann benutzt werden, um nur Daten des BAFU anzuzeigen:

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>

SELECT * WHERE {
    ?dataset a schema:Dataset;
        schema:name ?name;
        purl:creator <https://register.ld.admin.ch/opendataswiss/org/bundesamt-fur-umwelt-bafu>.
        
    FILTER(lang(?name) = 'de')
}

""")

display_result(df)

# Spezifische Datasets

In den nachfolgenden Abschnitten werden nun bestimmte Datasets näher untersucht und dabei wird die Struktur der eigentlichen Daten innerhalb der Datasets weiter erläutert. Zusätzlich wird aufgezeigt, wie georeferenzierte Daten direkt im Jupyter Notebook mit Hilfe einer Kartendarstellung angezeigt werden können.

## Qualität der Badegewässer

### Aktuellste Version des Datasets

Die erste Aufgabe ist, die aktuellste Version des entsprechenden Datasets herauszufinden. Ein Klick auf eine beliebige Version des Datensatzes (bspw. https://environment.ld.admin.ch/foen/ubd0104/3/) zeigt, dass dieser Datensatz und alle anderen einen gemeinsamen [purl:identifier](http://purl.org/dc/terms/identifier) "ubd0104" aufweisen. Somit kann folgende SPARQL Query mit einer [SPARQL subquery](https://en.wikibooks.org/wiki/SPARQL/Subqueries) Konstruktion das aktuellste Dataset bestimmen:

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>

SELECT * WHERE {

    ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version ?maxversion.
    {
        SELECT (max(?version) as ?maxversion) WHERE {
            ?dataset a schema:Dataset;
                purl:identifier "ubd0104";
                schema:version ?version.
        }
    }
}

""")

display_result(df)

### ObservationSet und Observation

Weil dieses Dataset vom [rdf:type](http://www.w3.org/1999/02/22-rdf-syntax-ns#type) [cube:Cube](https://cube.link/#Cube) ist, sind die eigentlichen Messwerte innerhalb eines [cube:ObservationSet](https://cube.link/#ObservationSet) zusammengefasst:

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>

SELECT ?observation WHERE {
    
    ?dataset a schema:Dataset;
        purl:identifier "ubd0104";
        schema:version ?maxversion;
        cube:observationSet/cube:observation ?observation.
    
    {
    SELECT (max(?version) as ?maxversion) WHERE {
        ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version ?version.
    }
    }
}

LIMIT 10

""")

display_result(df)

### Messwerttabelle

Um das verwendete Vokabular besser zu verstehen, kann eine einzelne Observation angeklickt werden und damit folgende Query erstellt werden:

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>
PREFIX ubd0104: <https://environment.ld.admin.ch/foen/ubd0104/>

SELECT ?name ?station ?dateOfProbing ?parameterType ?value WHERE {
    
    ?dataset a schema:Dataset;
        purl:identifier "ubd0104";
        schema:version ?maxversion;
        cube:observationSet/cube:observation ?observation.
    
    ?observation ubd0104:dateofprobing ?dateOfProbing;
        ubd0104:station ?station;
        ubd0104:parametertype ?parameterType;
        ubd0104:station ?station;
        ubd0104:value ?value.
    
    ?station schema:name ?name.
        
    {
    SELECT (max(?version) as ?maxversion) WHERE {
        ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version ?version.
    }
    }
}

LIMIT 100

""")

display_result(df)

### Kartendarstellung der Messstationen

Ein nächstes Ziel könnte sein, alle verfügbaren Messstationen auf einer Landkarte darzustellen. Die dazu notwendige SPARQL Query ist die folgende:

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>
PREFIX ubd0104: <https://environment.ld.admin.ch/foen/ubd0104/>


SELECT DISTINCT ?station ?canton ?name ?latitude ?longitude WHERE {
    
    ?dataset a schema:Dataset;
        purl:identifier "ubd0104";
        schema:version ?maxversion;
        cube:observationSet/cube:observation ?observation.
    
    ?observation ubd0104:station ?station.
    ?station ubd0104:kanton ?canton;
        schema:name ?name;
        schema:latitude ?latitude;
        schema:longitude ?longitude.
        
    {
    SELECT (max(?version) as ?maxversion) WHERE {
        ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version ?version.
    }
    }
}

""")

display_result(df)

Um die Kartendarstellung zu erzeugen, nutzen wir das Python Modul [folium](https://python-visualization.github.io/folium/), welches wir zuerst noch importieren müssen:

In [ ]:
# Karte erstellen und um das Mittel der Koordinaten zentrieren
m = folium.Map(location=[df['latitude'].mean(),df['longitude'].mean()], tiles="stamenterrain", zoom_start=8)

# Hinzufügen der Marker
for i in range(0,len(df)):
    folium.Marker(
        location=[df.iloc[i]['latitude'], df.iloc[i]['longitude']],
        popup=folium.Popup(folium.Html('<a href="' + df.iloc[i]['station'] + '" target="_blank">' + df.iloc[i]['name'] + '</a>', script=True), max_width=500),
        icon=folium.Icon(icon='person-swimming', prefix='fa')
   ).add_to(m)

#Karte anzeigen
m

### Zusätzliche Informationen in Karte

Um die Karte zu verbessern, wollen wir die Markierung in verschiedenen Farben anzeigen, die der Qualität der letzten Messungen entsprechen, und wenn man auf die Markierung klickt, sollte ein Popup mit zusätzlichen Informationen erscheinen.

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>
PREFIX ubd0104: <https://environment.ld.admin.ch/foen/ubd0104/>

SELECT ?name ?station ?latitude ?longitude ?date ?value WHERE {

    ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version 6;
            cube:observationSet/cube:observation ?observation.

        ?observation ubd0104:station ?station;
            ubd0104:parametertype "E.coli";
            ubd0104:dateofprobing ?date;
            ubd0104:value ?value.
        
        ?station schema:name ?name;
            schema:latitude ?latitude;
            schema:longitude ?longitude.
    

    {
    SELECT ?station (max(?date) as ?date) WHERE {

        ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version 6;
            cube:observationSet/cube:observation ?observation.

        ?observation ubd0104:station ?station;
            ubd0104:parametertype "E.coli";
            ubd0104:dateofprobing ?date.
    }

    GROUP BY ?station
    }
}
""")

display_result(df)

Jetzt soll die Karte erstellt werden:

In [ ]:
# Mapping von Qualität zu Farbe erstellen
conditions = [(df['value'] < 10),
              (df['value'] >= 10) & (df['value'] < 50),
              (df['value'] >= 50) & (df['value'] < 100),
              (df['value'] >= 100)
             ]

values = ["green", "darkgreen", "orange", "red"]

df['quality'] = np.select(conditions, values)

# Karte erstellen und um das Mittel der Koordinaten zentrieren
m = folium.Map(location=[df['latitude'].mean(),df['longitude'].mean()], tiles="stamenterrain", zoom_start=8)

# Hinzufügen der Marker
for i in range(0,len(df)):
    folium.Marker(
        location=[df.iloc[i]['latitude'], df.iloc[i]['longitude']],
        popup=folium.Popup(folium.Html('<p><a href="' + df.iloc[i]['station'] + '" target="_blank">' + df.iloc[i]['name'] + '</a></p>' + 
                                       '<p>Date: ' + df.iloc[i]['date'] + '</p>' + 
                                       '<p>Value E.Coli = ' + str(df.iloc[i]["value"]) + '</p>', script=True), max_width=500),
        icon=folium.Icon(icon='person-swimming', prefix='fa', color = df.iloc[i]['quality'])
   ).add_to(m)

# Karte anzeigen
m

## Public Transport

### LINDAS: Haltestellen öffentlicher Verkehr

In [ ]:
df = await query("""

SELECT *  
WHERE {
  ?station a <http://vocab.gtfs.org/terms#Station>.
} LIMIT 10

""")

display_result(df)

### Swisstopo: Bahnhöfe pro Gemeinde

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>

SELECT DISTINCT ?muni_name (COUNT(?stop) as ?number_of_stops) WHERE {
  ?stop a <http://vocab.gtfs.org/terms#Stop>;
      <https://geo.ld.admin.ch/def/transportation/meansOfTransportation> <https://geo.ld.admin.ch/vocabulary/MeansOfTransportation/B>;
      <http://schema.org/containedInPlace> ?muni.
    ?muni schema:name ?muni_name.
} GROUP BY ?muni_name ORDER BY DESC(?number_of_stops) LIMIT 20

""", "G")

display_result(df)

### Swisstopo: Gemeinden ohne ÖV Haltestellen

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?muni ?muni_name ?geometry WHERE {

    ?muni a <http://schema.org/AdministrativeArea>;
        <http://www.geonames.org/ontology#featureCode> <http://www.geonames.org/ontology#A.ADM3>;
        <http://purl.org/dc/terms/hasVersion> ?version;
        schema:name ?muni_name.
    ?version <http://purl.org/dc/terms/issued> "2023-01-01"^^xsd:date;
        <http://www.opengis.net/ont/geosparql#defaultGeometry>/<http://www.opengis.net/ont/geosparql#asWKT> ?geometry
        
    FILTER NOT EXISTS {
        ?stop a <http://vocab.gtfs.org/terms#Stop>;
            <http://schema.org/containedInPlace> ?muni.
    }

}

""", "G")

display_result(df[["muni", "muni_name"]])

In [ ]:
df = gpd.GeoDataFrame(df)

df["geometry"] = gpd.GeoSeries.from_wkt(df["geometry"], crs="EPSG:4326")

sw = df.bounds[["miny", "minx"]].min().values.tolist()
ne = df.bounds[["maxy", "maxx"]].max().values.tolist()

display(df.head())

In [ ]:
geo_json = df.to_json()

m = folium.Map(location=[47, 7], tiles='openstreetmap', zoom_start=10)

folium.GeoJson(geo_json, 
               name="geojson",
               style_function = lambda feature: {
                    "fillOpacity": 0.7,
                    "line_opacity": 0.5,
                    "weight": 1,
                    "fillColor": "blue"
               },
               tooltip=folium.features.GeoJsonTooltip(
                     fields=["muni_name"],
                     style=("background-color: white; color: #333333; font-family: arial; font-size: 12px; padding: 10px;"),
                     sticky=False
               )
).add_to(m)
folium.LayerControl().add_to(m)
m.fit_bounds([sw, ne])
m

### Swisstopo: Veraltete Daten

Folgende ÖV Haltestellen sind aktuell einer nicht mehr existierenden Gemeinde zugeordnet:

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX purl: <http://purl.org/dc/terms/>

SELECT ?stop ?stop_name ?numi_name WHERE {

    ?stop a <http://vocab.gtfs.org/terms#Stop>;
        schema:name ?stop_name;
        schema:containedInPlace ?muni.
    ?muni schema:name ?numi_name.
    
     FILTER NOT EXISTS {
        ?muni purl:hasVersion ?version.    
        ?version purl:issued "2023-01-01"^^xsd:date;
    }
}

""", "G")

display_result(df)

# Choropleth Karten

Choropleth Karten sind beliebte Mittel, einen Messwert für ein bestimmtes geografisches gebiet zu visualisieren. 

## Bevölkerungsdichte von Schweizer Gemeinden

Nachfolgend sollen Daten zur Bevölkerungsdichte visualisiert werden, die aus der Fläche und Anzahl Einwohner einer Gemeinde berechnet werden:

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX gn: <http://www.geonames.org/ontology#>
PREFIX ogis: <http://www.opengis.net/ont/geosparql#>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?bfs ?name ?version ?population ?area ?geometry WHERE {

?muni gn:featureCode gn:A.ADM3;
    schema:name ?name;
    schema:about ?about;
    <https://geo.ld.admin.ch/def/bfsNumber> ?bfs;
    purl:hasVersion ?version.
    
?version purl:issued "2023-01-01"^^xsd:date;
    ogis:defaultGeometry/ogis:asWKT ?geometry;
    gn:population ?population;
    <http://dbpedia.org/property/area> ?area;
    gn:parentADM1 ?canton.
?canton schema:name "Vaud".
    
}

""", "G")

df = gpd.GeoDataFrame(df)

df["geometry"] = gpd.GeoSeries.from_wkt(df["geometry"], crs="EPSG:4326")

df["density"] = df["population"] / df["area"]

sw = df.bounds[["miny", "minx"]].min().values.tolist()
ne = df.bounds[["maxy", "maxx"]].max().values.tolist()

display(df.head())

In [ ]:
df["html"] = "<a href='" + df["version"] + "' target='_blank'>" + df["name"] + "</a>"

geo_json = df.to_json()

colorscale = branca.colormap.linear.YlOrRd_09.scale(0, 10)

m = folium.Map(location=[47, 7], tiles='openstreetmap', zoom_start=10)

folium.GeoJson(geo_json, 
               name="geojson",
               style_function = lambda feature: {
                    "fillOpacity": 0.7,
                    "line_opacity": 0.5,
                    "weight": 1,
                    "fillColor": colorscale(feature["properties"]["density"])
               },
               highlight_function=lambda feature: {
                     "fillColor": colorscale(feature["properties"]["density"]),
                     "fillOpacity": 0.2
               },
               tooltip=folium.features.GeoJsonTooltip(
                     fields=["name", "population", "area", "density"],
                     style=("background-color: white; color: #333333; font-family: arial; font-size: 12px; padding: 10px;"),
                     sticky=False
               ),
               popup=folium.features.GeoJsonPopup(
                    fields=["html", "population"],
                    aliases=["Gemeinde", "Einwohner"],
                    localize=False,
                    labels=True
               )
).add_to(m)
folium.LayerControl().add_to(m)
m.fit_bounds([sw, ne])
m